# In this lab, we will learn to use the python packet manipulation library Scapy to modify Ethernet frames.

Scapy (https://scapy.net/) is a python library that enables manipulating packets.  Recall that network communication is more than just the ability to send 1s and 0s. We need structure so that both sides of the communication can understand each other.  In Ethernet, this structure was: 

\[ dst \]\[ src \]\[ type \]\[ payload \]\[ FCS \]

This naturally can translate to a data structure in a programming language like Python.  With Scapy, you can read packets from a file, read live traffic from the network interface, or generate packets from scratch.  

For a full tutorial, you can visit the scapy website.  Here, we'll go through just enough to complete this lab.

Useful cheat sheet for scapy:
https://wiki.sans.blue/Tools/pdfs/ScapyCheatSheet_v0.2.pdf


This lab, as typical for Jupyter notebooks, contains multiple cells which you can execute one at a time and then see the result of each cell's execution.  The execution state is shared across cells so the results of one cell can be seen by a subsequent cell.

Execute each cell below, in sequence, to see how you can interact with Scapy.  Then provide your own implementation for the cell containing '# your code here' to complete this lab.  You will be graded on the result of that implementation.

In [1]:
#We need to import the scapy library, in this case everything (*)
from scapy.all import *

#Read in a trace of packets from a file
pkts = rdpcap("intro-wireshark-trace1.pcap")

The pkts variable is now an array of the Packet data structure which was populated in the operation above from reading the pcap file. 

In [2]:
pkt = pkts[1]

# Do checks on what types are in the packet
eth = Ether in pkt
print(eth)

# You can get the name of the layers in the packet (requesting a specific one)
layer = pkt.getlayer(0)
print(layer)

# or, there's a variable within the pkt structure indicating the name of the type
exp = pkt.name
print (exp)

# You can list all the available protocols - won't, since it's large
#ls()

# Or list the fields in a particular layer
ls(Ether)

True
Ether / IPv6 / ICMPv6ND_RA / ICMPv6NDOptRDNSS / ICMPv6NDOptPrefixInfo / ICMPv6NDOptRouteInfo / ICMPv6 Neighbor Discovery Option - Source Link-Layer Address 00:50:f1:80:00:00
Ethernet
dst        : DestMACField                        = ('None')
src        : SourceMACField                      = ('None')
type       : XShortEnumField                     = ('36864')


In [3]:
# Dump the packet contents in hex
hexdump(pkt)

0000  33 33 00 00 00 01 00 50 F1 80 00 00 86 DD 60 00  33.....P......`.
0010  00 00 00 78 3A FF FE 80 00 00 00 00 00 00 A4 F1  ...x:...........
0020  94 FF FE C5 00 4E FF 02 00 00 00 00 00 00 00 00  .....N..........
0030  00 00 00 00 00 01 86 00 A6 66 40 C0 00 B4 00 00  .........f@.....
0040  00 00 00 00 00 00 19 05 00 00 00 01 51 80 20 01  ............Q. .
0050  05 58 FE ED 00 00 00 00 00 00 00 00 00 01 20 01  .X............ .
0060  05 58 FE ED 00 00 00 00 00 00 00 00 00 02 03 04  .X..............
0070  40 C0 00 04 A5 18 00 04 A5 18 00 00 00 00 26 01  @.............&.
0080  01 93 83 00 63 B5 00 00 00 00 00 00 00 00 18 03  ....c...........
0090  00 00 00 00 00 B4 00 00 00 00 00 00 00 00 00 00  ................
00a0  00 00 00 00 00 00 01 01 00 50 F1 80 00 00        .........P....


In [4]:
# Dump the packet in a nice format
pkt.show()

###[ Ethernet ]### 
  dst       = 33:33:00:00:00:01
  src       = 00:50:f1:80:00:00
  type      = IPv6
###[ IPv6 ]### 
     version   = 6
     tc        = 0
     fl        = 0
     plen      = 120
     nh        = ICMPv6
     hlim      = 255
     src       = fe80::a4f1:94ff:fec5:4e
     dst       = ff02::1
###[ ICMPv6 Neighbor Discovery - Router Advertisement ]### 
        type      = Router Advertisement
        code      = 0
        cksum     = 0xa666
        chlim     = 64
        M         = 1
        O         = 1
        H         = 0
        prf       = Medium (default)
        P         = 0
        res       = 0
        routerlifetime= 180
        reachabletime= 0
        retranstimer= 0
###[ ICMPv6 Neighbor Discovery Option - Recursive DNS Server Option ]### 
           type      = 25
           len       = 5
           res       = 0
           lifetime  = 86400
           dns       = [ 2001:558:feed::1, 2001:558:feed::2 ]
###[ ICMPv6 Neighbor Discovery Option - Prefix Informa

In [ ]:
# You can manipulate the packet through fields

pkt.dst = "11:22:33:44:55:66"

pkt.show()

Your task for the graded portion of this lab is to iterate (loop) over all the packets loaded into the pkts array from the first cell and for each packet, ONLY if it includes Ether, change its source address to 11:11:11:11:11:11 and its destination address to 22:22:22:22:22:22. Do not add to or remove packets from the array.

In [10]:
# your code here
# Counter to track how many packets were modified
modified_count = 0
total_ether_count = 0


for i in range(len(pkts)):
    if Ether in pkts[i]:        
        total_ether_count += 1
        pkts[i].src = "11:11:11:11:11:11"
        pkts[i].dst = "22:22:22:22:22:22"
        
        modified_count += 1

        
all_modified = True
for i in range(len(pkts)):
    if Ether in pkts[i]:
        if pkts[i].src != "11:11:11:11:11:11" or pkts[i].dst != "22:22:22:22:22:22":
            all_modified = False
            print(f"Packet {i} was not properly modified!")
            break

if all_modified:
    print(f"Success! All {modified_count} Ethernet packets were modified correctly.")
    print(f"Total Ethernet packets: {total_ether_count}")
else:
    print("Failed to modify all Ethernet packets.")



Success! All 638 Ethernet packets were modified correctly.
Total Ethernet packets: 638


In [8]:
"""This cell contains hidden tests that will be executed by the autograder.
You do not need to modify or execute this cell yourself.
Submit the assignment to generate your grade.
If you didn't get 100%, review the task description for the cell above and check that your implementation is correct.
"""



"This cell contains hidden tests that will be executed by the autograder.\nYou do not need to modify or execute this cell yourself.\nSubmit the assignment to generate your grade.\nIf you didn't get 100%, review the task description for the cell above and check that your implementation is correct.\n"